# 📈 AIOps — Session 9
# Anomaly Detection in IT Operations

---

| | |
|---|---|
| **Course** | AIOps Engineering |
| **Lab** | Lab 9 |
| **Topics** | Threshold-Based vs ML Anomalies · Multivariate Detection · Kubernetes Infrastructure Anomalies |
| **Duration** | ~90 minutes |
| **Platform** | Kubernetes (kubectl must be available in this JupyterLab environment) |

---

## 🎯 Learning Objectives

By completing this lab you will be able to:

1. **Explain** the limitations of static threshold alerting and why they cause both false positives and false negatives
2. **Implement** Z-Score and IQR-based univariate anomaly detection on real metric time-series
3. **Apply** Isolation Forest and DBSCAN for multivariate anomaly detection across correlated metrics
4. **Use** LSTM Autoencoders to detect subtle temporal anomalies that statistical methods miss
5. **Detect** Kubernetes infrastructure anomalies (CPU throttling, OOMKills, pod restarts, network saturation)
6. **Compare** all approaches on the same dataset and choose the right tool for each scenario

---

## 🗺️ Lab Architecture

We simulate a **Kubernetes-hosted e-commerce platform** with a full Prometheus-style metric stack:

```
  Kubernetes Cluster
  ├── frontend          (HTTP, user traffic, 2 replicas)
  ├── api-gateway       (request routing, rate limiting)
  ├── order-service     (business logic, DB writes)
  ├── payment-service   (external API calls)
  └── inventory-service (DB reads, cache)

  Metrics collected per service per minute:
  ├── cpu_usage_pct          ├── memory_usage_pct
  ├── http_request_rate      ├── http_error_rate_pct
  ├── p99_latency_ms         ├── pod_restarts
  └── network_bytes_out      └── gc_pause_ms
```

---

## 📋 Lab Phases

| Phase | Title | Key Concept |
|-------|-------|-------------|
| **0** | Setup & K8s Deployment | Environment |
| **1** | Metric Data Generation | Realistic Time-Series with Injected Anomalies |
| **2** | Threshold-Based Detection | Static Thresholds — The Old Way |
| **3** | Statistical Detection | Z-Score & IQR — Univariate ML |
| **4** | Isolation Forest | Multivariate Anomaly Detection |
| **5** | DBSCAN Clustering | Density-Based Anomalies |
| **6** | LSTM Autoencoder | Deep Learning for Temporal Anomalies |
| **7** | Kubernetes-Specific Anomalies | OOMKill · Throttling · Pod Churn |
| **8** | Method Comparison | Which Detector Wins? |
| **9** | Cleanup | kubectl delete |

> ▶️ **Run each cell in order using `Shift + Enter`**

---
## Phase 0 — Environment Setup & Kubernetes Deployment

Install libraries, verify `kubectl`, and deploy the workload namespace we will monitor.

In [ ]:
# ── Install required Python libraries ─────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q \
    pandas numpy matplotlib seaborn scikit-learn \
    tensorflow keras tabulate ipywidgets
print("✅ Libraries installed")

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import subprocess, time, warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

# TensorFlow / Keras for LSTM autoencoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)
tf.random.set_seed(42)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor':   '#ffffff',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        10,
})

COLORS = {
    'normal':   '#2ecc71',
    'anomaly':  '#e74c3c',
    'warning':  '#f39c12',
    'info':     '#3498db',
    'critical': '#8e44ad',
}

def run(cmd):
    """Execute shell command and print output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.returncode != 0 and r.stderr.strip(): print('STDERR:', r.stderr.strip())
    return r.stdout.strip(), r.returncode

print('✅ Imports ready')
print(f'   TensorFlow {tf.__version__} · scikit-learn ready')

In [ ]:
# ── Verify kubectl access ──────────────────────────────────────────────────────
print('Cluster nodes:')
print('-' * 60)
out, rc = run('kubectl get nodes')

if rc == 0:
    print('\n✅ kubectl connected — ready to proceed!')
else:
    print('\n❌ kubectl cannot reach the cluster.')
    print('   Check your KUBECONFIG before continuing.')

In [ ]:
# ── Deploy the monitoring target workloads on Kubernetes ──────────────────────
# These are the services whose metrics we will analyse for anomalies.
# In a real environment these would already exist; here we create them
# so the lab is fully self-contained.

NAMESPACE = 'aiops-lab9'

SERVICES = [
    ('frontend',          3000, 2, 'frontend'),
    ('api-gateway',       8080, 2, 'gateway'),
    ('order-service',     8081, 2, 'backend'),
    ('payment-service',   8082, 1, 'backend'),
    ('inventory-service', 8083, 1, 'backend'),
]

def svc_manifest(name, port, replicas, tier):
    return f"""
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {name}
  namespace: {NAMESPACE}
  labels:
    app: {name}
    tier: {tier}
spec:
  replicas: {replicas}
  selector:
    matchLabels:
      app: {name}
  template:
    metadata:
      labels:
        app: {name}
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "{port}"
    spec:
      containers:
      - name: {name}
        image: nginx:alpine
        ports:
        - containerPort: 80
        resources:
          requests: {{cpu: "25m", memory: "32Mi"}}
          limits:   {{cpu: "200m", memory: "256Mi"}}
        livenessProbe:
          httpGet: {{path: "/", port: 80}}
          initialDelaySeconds: 5
          periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: {name}
  namespace: {NAMESPACE}
spec:
  selector:
    app: {name}
  ports:
  - port: {port}
    targetPort: 80
"""

ns_yaml = f'apiVersion: v1\nkind: Namespace\nmetadata:\n  name: {NAMESPACE}\n  labels:\n    lab: session9\n'
manifests = ns_yaml + ''.join(svc_manifest(*s) for s in SERVICES)

with open('/tmp/lab9-deploy.yaml', 'w') as f:
    f.write(manifests)

print('Applying Kubernetes manifests...')
run('kubectl apply -f /tmp/lab9-deploy.yaml')
print('\n⏳ Waiting 25 seconds for pods...')
time.sleep(25)

print('\nDeployment status:')
run(f'kubectl get pods -n {NAMESPACE}')
print('\n✅ Workloads deployed. We will now simulate their metric streams.')

---
## Phase 1 — Realistic Metric Data Generation

### 📖 Concept: What Good Metric Data Looks Like

In production, Prometheus scrapes metrics every 15–60 seconds. A typical service produces:

- **Diurnal pattern** — traffic peaks during business hours, troughs at night
- **Weekly seasonality** — weekends differ from weekdays
- **Correlated metrics** — CPU and request rate rise together; memory may lag
- **Occasional spikes** — deployments, cron jobs, batch processing
- **True anomalies** — memory leaks, traffic floods, resource exhaustion

We generate **7 days of 1-minute resolution metrics** for 5 services — that's **50,400 data points** — with **precisely labelled anomalies** injected at known times so we can measure detection accuracy.

In [ ]:
# ── Metric generation helpers ──────────────────────────────────────────────────

SIM_START   = datetime(2025, 6, 9, 0, 0, 0)   # Monday 00:00
SIM_DAYS    = 7
FREQ_MIN    = 1                                  # 1-minute resolution
N_POINTS    = SIM_DAYS * 24 * 60               # 10,080 per service
SVC_NAMES   = [s[0] for s in SERVICES]

timestamps  = [SIM_START + timedelta(minutes=i) for i in range(N_POINTS)]
t           = np.arange(N_POINTS)

def diurnal(t, peak_hour=14, width=4):
    """Smooth diurnal (daily) pattern — peak at peak_hour, trough at night."""
    hour_of_day = (t % (24 * 60)) / 60          # fractional hour 0–24
    return np.exp(-0.5 * ((hour_of_day - peak_hour) / width) ** 2)

def weekday_factor(t):
    """Business-day modulation: weekdays 1.0, weekends 0.35."""
    day_of_week = (t // (24 * 60)) % 7          # 0=Mon … 6=Sun
    return np.where(day_of_week < 5, 1.0, 0.35)

def add_noise(signal, noise_pct=0.05):
    return signal + np.random.normal(0, noise_pct * signal.mean(), len(signal))

# ── Baseline load profile ──────────────────────────────────────────────────────
BASE_LOAD = diurnal(t) * weekday_factor(t)   # shape: (N_POINTS,)

print(f'Generating {N_POINTS:,} time-series points per service across {len(SVC_NAMES)} services...')
print(f'Total dataset size: {N_POINTS * len(SVC_NAMES):,} metric observations')

In [ ]:
# ── Define injected anomaly events ─────────────────────────────────────────────
# These are the GROUND TRUTH labels we will use to evaluate detector accuracy.
# Format: (start_minute, duration_minutes, service, type, description)

# Day offsets: Mon=0, Tue=1440, Wed=2880, Thu=4320, Fri=5760
ANOMALY_EVENTS = [
    # (start_min,  duration, service,            type,               description)
    (1440 + 600,   45,  'order-service',     'memory_leak',      'Gradual memory leak — OOM in 45 min'),
    (1440 + 780,   20,  'api-gateway',       'traffic_spike',    'Flash sale traffic flood 10x normal'),
    (2880 + 480,   60,  'inventory-service', 'cpu_throttle',     'CPU throttling — pod hitting limit'),
    (2880 + 600,   30,  'payment-service',   'latency_spike',    'External payment API degradation'),
    (4320 + 120,   15,  'order-service',     'pod_restart',      'CrashLoopBackoff — 5 restarts'),
    (4320 + 480,   90,  'inventory-service', 'db_saturation',    'DB connection pool exhausted'),
    (4320 + 720,   10,  'frontend',          'traffic_spike',    'Bot traffic surge'),
    (5760 + 300,   120, 'api-gateway',       'memory_leak',      'Memory leak before weekend deploy'),
    (5760 + 540,   20,  'payment-service',   'network_anomaly',  'Network retransmits spike'),
    (6480 + 200,   25,  'frontend',          'cpu_throttle',     'Weekend batch job CPU spike'),
]

# Build ground-truth label arrays per service
gt_labels = {svc: np.zeros(N_POINTS, dtype=int) for svc in SVC_NAMES}
for (start, dur, svc, atype, desc) in ANOMALY_EVENTS:
    end = min(start + dur, N_POINTS)
    gt_labels[svc][start:end] = 1

total_anomalous = sum(v.sum() for v in gt_labels.values())
print(f'Injected anomaly events : {len(ANOMALY_EVENTS)}')
print(f'Total anomalous minutes : {total_anomalous:,} '
      f'({total_anomalous / (N_POINTS * len(SVC_NAMES)) * 100:.1f}% of dataset)')
print()
print('Anomaly schedule:')
rows = []
for (s, d, svc, atype, desc) in ANOMALY_EVENTS:
    rows.append([
        (SIM_START + timedelta(minutes=s)).strftime('%a %H:%M'),
        f'{d} min', svc, atype, desc
    ])
print(tabulate(rows, headers=['Start', 'Duration', 'Service', 'Type', 'Description'],
               tablefmt='rounded_outline'))

In [ ]:
# ── Generate full metric time-series for all services ─────────────────────────
# Each service has slightly different baseline characteristics.

SVC_PROFILES = {
    'frontend':          {'cpu_base': 35,  'mem_base': 50,  'rps_base': 800,  'lat_base': 80},
    'api-gateway':       {'cpu_base': 40,  'mem_base': 55,  'rps_base': 750,  'lat_base': 45},
    'order-service':     {'cpu_base': 55,  'mem_base': 60,  'rps_base': 300,  'lat_base': 150},
    'payment-service':   {'cpu_base': 30,  'mem_base': 45,  'rps_base': 120,  'lat_base': 250},
    'inventory-service': {'cpu_base': 45,  'mem_base': 65,  'rps_base': 400,  'lat_base': 60},
}

def generate_service_metrics(svc, profile, t, base_load):
    """Generate 8 metrics for one service with realistic correlations."""
    rng = np.random.default_rng(seed=abs(hash(svc)) % 10000)

    # ── Normal baseline metrics ───────────────────────────────────────────────
    rps     = add_noise(base_load * profile['rps_base'] + 20, 0.08)
    cpu     = add_noise(base_load * profile['cpu_base'] + 10, 0.06)
    mem     = add_noise(np.full(N_POINTS, profile['mem_base']) +
                        np.cumsum(rng.normal(0, 0.01, N_POINTS)).clip(-10, 10), 0.03)
    latency = add_noise(base_load * profile['lat_base'] + 20, 0.10)
    errors  = add_noise(np.maximum(0, (cpu - 70) * 0.05 + rng.normal(0, 0.1, N_POINTS)), 0.5)
    net_out = add_noise(rps * rng.uniform(1.2, 1.8), 0.07)
    restarts= np.zeros(N_POINTS)
    gc_ms   = add_noise(base_load * 15 + 5, 0.15)

    # Clip to realistic ranges
    cpu     = np.clip(cpu, 2, 98)
    mem     = np.clip(mem, 10, 98)
    rps     = np.clip(rps, 0, None)
    latency = np.clip(latency, 1, None)
    errors  = np.clip(errors, 0, 100)
    gc_ms   = np.clip(gc_ms, 0, None)

    # ── Inject anomalies for this service ────────────────────────────────────
    for (start, dur, ev_svc, atype, _) in ANOMALY_EVENTS:
        if ev_svc != svc:
            continue
        end   = min(start + dur, N_POINTS)
        prog  = np.linspace(0, 1, end - start)   # 0→1 progression

        if atype == 'memory_leak':
            mem[start:end]     += prog * 35           # memory climbs steadily
            gc_ms[start:end]   += prog * 120          # GC pressure rises
            cpu[start:end]     += prog * 15

        elif atype == 'traffic_spike':
            spike = np.exp(-((prog - 0.2) / 0.15)**2) * 8  # bell-curve spike
            rps[start:end]     += spike * profile['rps_base']
            cpu[start:end]     += spike * 35
            latency[start:end] += spike * profile['lat_base'] * 2
            errors[start:end]  += spike * 5

        elif atype == 'cpu_throttle':
            cpu[start:end]     = np.clip(cpu[start:end] + 40 + rng.normal(0, 3, end-start), 0, 100)
            latency[start:end] += 80 + rng.normal(0, 10, end-start)

        elif atype == 'latency_spike':
            latency[start:end] += 600 + rng.normal(0, 50, end-start)
            errors[start:end]  += 8 + rng.normal(0, 2, end-start)

        elif atype == 'pod_restart':
            restarts[start:start+5] = [1, 0, 1, 0, 2]
            cpu[start:end]     = rng.uniform(2, 20, end-start)   # low CPU = restarting
            errors[start:end]  += rng.uniform(15, 40, end-start)

        elif atype == 'db_saturation':
            latency[start:end] += prog * 400
            errors[start:end]  += prog * 20
            cpu[start:end]     += 20 + rng.normal(0, 5, end-start)

        elif atype == 'network_anomaly':
            net_out[start:end] *= 4 + rng.normal(0, 0.3, end-start)
            latency[start:end] += 200

    # Final clip
    cpu     = np.clip(cpu, 0, 100)
    mem     = np.clip(mem, 0, 100)
    errors  = np.clip(errors, 0, 100)

    return pd.DataFrame({
        'timestamp':        timestamps,
        'service':          svc,
        'cpu_pct':          np.round(cpu, 2),
        'mem_pct':          np.round(mem, 2),
        'rps':              np.round(np.abs(rps), 1),
        'error_rate_pct':   np.round(np.abs(errors), 3),
        'p99_latency_ms':   np.round(np.abs(latency), 1),
        'pod_restarts':     restarts.astype(int),
        'net_bytes_out_kb': np.round(np.abs(net_out), 1),
        'gc_pause_ms':      np.round(np.abs(gc_ms), 1),
        'is_anomaly':       gt_labels[svc],
    })

print('Generating metric time-series for all services...')
dfs = {}
for svc in SVC_NAMES:
    dfs[svc] = generate_service_metrics(svc, SVC_PROFILES[svc], t, BASE_LOAD.copy())
    print(f'  {svc:25s}: {len(dfs[svc]):,} rows  (anomalous: {dfs[svc]["is_anomaly"].sum():,})')

df_all = pd.concat(dfs.values(), ignore_index=True)
print(f'\nTotal dataset: {len(df_all):,} rows × {len(df_all.columns)} columns')

In [ ]:
# ── Exploratory visualisation: 7-day metric overview ─────────────────────────
# Focus on order-service which has the richest anomaly set.
svc_demo = 'order-service'
df_demo  = dfs[svc_demo].copy()
df_demo['hour'] = (df_demo['timestamp'] - SIM_START).dt.total_seconds() / 3600

METRICS_DISPLAY = [
    ('cpu_pct',         'CPU Usage (%)',         '#e74c3c'),
    ('mem_pct',         'Memory Usage (%)',       '#9b59b6'),
    ('rps',             'Requests / second',      '#3498db'),
    ('p99_latency_ms',  'P99 Latency (ms)',        '#e67e22'),
    ('error_rate_pct',  'Error Rate (%)',           '#c0392b'),
    ('gc_pause_ms',     'GC Pause (ms)',            '#1abc9c'),
]

fig, axes = plt.subplots(len(METRICS_DISPLAY), 1, figsize=(16, 14), sharex=True)
fig.suptitle(f'7-Day Metric Overview — {svc_demo}  (red bands = injected anomalies)',
             fontsize=13, fontweight='bold')

anomaly_intervals = [
    ((s - (SIM_START - SIM_START).total_seconds()/60) / 60,
     (s + d) / 60)
    for (s, d, svc, _, _) in ANOMALY_EVENTS if svc == svc_demo
]

for ax, (metric, ylabel, color) in zip(axes, METRICS_DISPLAY):
    ax.plot(df_demo['hour'], df_demo[metric], color=color, linewidth=0.6, alpha=0.85)
    # Shade anomaly windows
    for (a_start_h, a_end_h) in anomaly_intervals:
        ax.axvspan(a_start_h, a_end_h, alpha=0.2, color='red', label='_nolegend_')
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xlim(0, SIM_DAYS * 24)

# X-axis: day labels
axes[-1].set_xlabel('Time (hours from Monday 00:00)')
day_ticks = [d * 24 for d in range(SIM_DAYS + 1)]
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', '']
axes[-1].set_xticks(day_ticks)
axes[-1].set_xticklabels(day_labels)

# Add day lines
for ax in axes:
    for d in range(1, SIM_DAYS):
        ax.axvline(d * 24, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)

red_patch = mpatches.Patch(color='red', alpha=0.3, label='Injected anomaly window')
axes[0].legend(handles=[red_patch], loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/phase1_metric_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\n💡 Notice: Red bands mark injected anomalies — subtle in some metrics, obvious in others.')
print('   Threshold-based detectors will miss subtle ones. ML detectors are needed.')

---
## Phase 2 — Threshold-Based Detection: The Old Way 🚦

### 📖 Concept: Why Static Thresholds Fail

Traditional monitoring sets **static thresholds** like:
- Alert if `cpu > 80%`
- Alert if `latency > 500ms`
- Alert if `error_rate > 5%`

**Problems with this approach:**

| Problem | Example |
|---------|------|
| **False positives** | CPU hits 82% during a normal Monday morning traffic peak |
| **False negatives** | A memory leak slowly climbing from 60% → 79% never triggers |
| **Context blindness** | 500ms latency is fine for a batch job, catastrophic for checkout |
| **No seasonality** | Same threshold for 3am (quiet) and 3pm (peak) |
| **Metric correlation ignored** | CPU=75% alone is fine; CPU=75% + errors=8% + latency=2x = incident |

In [ ]:
# ── Implement static threshold detection ──────────────────────────────────────
THRESHOLDS = {
    'cpu_pct':          80.0,   # %
    'mem_pct':          85.0,   # %
    'error_rate_pct':   5.0,    # %
    'p99_latency_ms':   500.0,  # ms
    'pod_restarts':     1,      # count
}

def threshold_detect(df, thresholds):
    """Return boolean Series: True = at least one metric exceeds its threshold."""
    flagged = pd.Series(False, index=df.index)
    for metric, limit in thresholds.items():
        if metric in df.columns:
            flagged |= df[metric] > limit
    return flagged.astype(int)

# Evaluate on all services
results_thresh = {}
for svc in SVC_NAMES:
    df = dfs[svc].copy()
    df['thresh_pred'] = threshold_detect(df, THRESHOLDS)
    results_thresh[svc] = df

# ── Compute detection metrics ─────────────────────────────────────────────────
def detection_metrics(y_true, y_pred, method_name, svc_name):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    fpr       = fp / (fp + tn + 1e-9)
    return {
        'method': method_name, 'service': svc_name,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Precision': round(precision, 3),
        'Recall':    round(recall, 3),
        'F1':        round(f1, 3),
        'FPR':       round(fpr, 4),
    }

thresh_scores = []
for svc, df in results_thresh.items():
    thresh_scores.append(detection_metrics(df['is_anomaly'], df['thresh_pred'], 'Threshold', svc))

df_thresh_scores = pd.DataFrame(thresh_scores)
print('Threshold Detection Results:')
print(tabulate(
    df_thresh_scores[['service','TP','FP','FN','Precision','Recall','F1','FPR']],
    headers='keys', tablefmt='rounded_outline', showindex=False
))
print()
avg = df_thresh_scores[['Precision','Recall','F1','FPR']].mean()
print(f'Average  Precision={avg["Precision"]:.3f}  Recall={avg["Recall"]:.3f}  '
      f'F1={avg["F1"]:.3f}  FPR={avg["FPR"]:.4f}')

In [ ]:
# ── Visualise threshold failures: False positives vs False negatives ──────────
# Zoom into order-service for clarity
df_demo2 = results_thresh['order-service'].copy()
df_demo2['hour'] = (df_demo2['timestamp'] - SIM_START).dt.total_seconds() / 3600

# Focus window: Tuesday (hours 24–48) — where memory leak anomaly occurs
mask_window = (df_demo2['hour'] >= 24) & (df_demo2['hour'] <= 48)
dw = df_demo2[mask_window].copy()

fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
fig.suptitle('Phase 2 — Threshold Detection: False Positives & False Negatives (order-service, Tuesday)',
             fontsize=12, fontweight='bold')

# Panel 1: CPU with threshold line
ax = axes[0]
ax.plot(dw['hour'], dw['cpu_pct'], color='#3498db', linewidth=0.8, label='CPU %')
ax.axhline(THRESHOLDS['cpu_pct'], color='red', linestyle='--', linewidth=1.5, label=f'Threshold={THRESHOLDS["cpu_pct"]}%')
fp_mask = (dw['thresh_pred'] == 1) & (dw['is_anomaly'] == 0)
ax.scatter(dw[fp_mask]['hour'], dw[fp_mask]['cpu_pct'],
           color='orange', s=15, zorder=5, label='False Positive')
ax.set_ylabel('CPU %')
ax.set_title('CPU — Normal traffic peaks trigger false positives')
ax.legend(fontsize=8, loc='upper right')

# Panel 2: Memory with threshold line — memory leak is BELOW threshold
ax2 = axes[1]
ax2.plot(dw['hour'], dw['mem_pct'], color='#9b59b6', linewidth=0.8, label='Memory %')
ax2.axhline(THRESHOLDS['mem_pct'], color='red', linestyle='--', linewidth=1.5,
            label=f'Threshold={THRESHOLDS["mem_pct"]}%')
# Shade the true anomaly window
for (s, d, svc, atype, _) in ANOMALY_EVENTS:
    if svc == 'order-service' and atype == 'memory_leak':
        ax2.axvspan(s/60, (s+d)/60, alpha=0.2, color='red', label='True anomaly')
fn_mask = (dw['thresh_pred'] == 0) & (dw['is_anomaly'] == 1)
ax2.scatter(dw[fn_mask]['hour'], dw[fn_mask]['mem_pct'],
            color='purple', s=15, zorder=5, label='False Negative (missed!)')
ax2.set_ylabel('Memory %')
ax2.set_title('Memory — Gradual leak MISSED because it never crosses threshold')
ax2.legend(fontsize=8, loc='upper left')

# Panel 3: Ground truth vs threshold prediction
ax3 = axes[2]
ax3.fill_between(dw['hour'], dw['is_anomaly'], alpha=0.5,
                 color=COLORS['anomaly'], label='Ground truth anomaly')
ax3.fill_between(dw['hour'], dw['thresh_pred'] * 0.7, alpha=0.5,
                 color=COLORS['warning'], label='Threshold prediction')
ax3.set_ylabel('Anomaly flag')
ax3.set_xlabel('Hour of week')
ax3.set_title('Ground Truth vs Threshold Predictions — Notice the misalignment')
ax3.legend(fontsize=8)
ax3.set_yticks([0, 0.7, 1])
ax3.set_yticklabels(['Normal', 'Predicted', 'True Anomaly'])

plt.tight_layout()
plt.savefig('/tmp/phase2_threshold.png', dpi=120, bbox_inches='tight')
plt.show()
print()
print('💡 KEY INSIGHT: Threshold detection has two fundamental failure modes:')
print('   1. FALSE POSITIVES  — CPU spikes at peak hours trigger alerts on healthy systems')
print('   2. FALSE NEGATIVES  — Gradual memory leaks never cross the threshold until it is too late')

---
## Phase 3 — Statistical Anomaly Detection: Z-Score & IQR 📊

### 📖 Concept: Dynamic Baselines with Statistics

Instead of a fixed threshold, statistical methods compute a **dynamic baseline** from recent history and flag deviations from that baseline.

| Method | Formula | Strength | Weakness |
|--------|---------|----------|----------|
| **Z-Score** | `z = (x - μ) / σ` — flag if `|z| > threshold` | Simple, fast, well-understood | Assumes Gaussian distribution |
| **IQR** | Flag if `x < Q1 - 1.5*IQR` or `x > Q3 + 1.5*IQR` | Robust to outliers | Less sensitive to gradual drifts |
| **Rolling Z-Score** | Z-score on a rolling window | Adapts to seasonality | Window size is a tuning parameter |

We apply **rolling Z-score** (window = 2 hours = 120 min) so the baseline adapts to diurnal patterns.

In [ ]:
# ── Rolling Z-Score detector ───────────────────────────────────────────────────
ZSCORE_WINDOW  = 120    # rolling window (minutes)
ZSCORE_THRESH  = 3.0    # flag if |z| > this
METRICS_DETECT = ['cpu_pct', 'mem_pct', 'p99_latency_ms', 'error_rate_pct', 'gc_pause_ms']

def rolling_zscore_detect(df, metrics, window, threshold):
    df = df.copy()
    z_scores = pd.DataFrame(index=df.index)

    for m in metrics:
        roll_mean = df[m].rolling(window=window, min_periods=10).mean()
        roll_std  = df[m].rolling(window=window, min_periods=10).std()
        z = (df[m] - roll_mean) / (roll_std + 1e-9)
        z_scores[m] = z
        df[f'zscore_{m}'] = z.round(3)

    # Anomaly = any metric has |z| > threshold
    df['zscore_pred'] = (z_scores.abs() > threshold).any(axis=1).astype(int)
    return df

results_zscore = {}
for svc in SVC_NAMES:
    results_zscore[svc] = rolling_zscore_detect(
        dfs[svc].copy(), METRICS_DETECT, ZSCORE_WINDOW, ZSCORE_THRESH
    )

zscore_scores = []
for svc, df in results_zscore.items():
    zscore_scores.append(detection_metrics(df['is_anomaly'], df['zscore_pred'], 'Z-Score', svc))

df_zscores = pd.DataFrame(zscore_scores)
print('Rolling Z-Score Detection Results (window=120min, threshold=3σ):')
print(tabulate(
    df_zscores[['service','TP','FP','FN','Precision','Recall','F1','FPR']],
    headers='keys', tablefmt='rounded_outline', showindex=False
))
avg_z = df_zscores[['Precision','Recall','F1']].mean()
print(f'\nAverage  Precision={avg_z["Precision"]:.3f}  Recall={avg_z["Recall"]:.3f}  F1={avg_z["F1"]:.3f}')

In [ ]:
# ── IQR-based detector ─────────────────────────────────────────────────────────
IQR_WINDOW = 240   # 4-hour rolling window for IQR

def rolling_iqr_detect(df, metrics, window):
    df = df.copy()
    flags = pd.DataFrame(index=df.index)

    for m in metrics:
        roll_q1  = df[m].rolling(window=window, min_periods=20).quantile(0.25)
        roll_q3  = df[m].rolling(window=window, min_periods=20).quantile(0.75)
        roll_iqr = roll_q3 - roll_q1
        lower    = roll_q1 - 1.5 * roll_iqr
        upper    = roll_q3 + 1.5 * roll_iqr
        flags[m] = ((df[m] < lower) | (df[m] > upper)).astype(int)

    df['iqr_pred'] = flags.any(axis=1).astype(int)
    return df

results_iqr = {}
for svc in SVC_NAMES:
    results_iqr[svc] = rolling_iqr_detect(dfs[svc].copy(), METRICS_DETECT, IQR_WINDOW)

iqr_scores = []
for svc, df in results_iqr.items():
    iqr_scores.append(detection_metrics(df['is_anomaly'], df['iqr_pred'], 'IQR', svc))

df_iqr_scores = pd.DataFrame(iqr_scores)
print('IQR Detection Results (window=240min):')
print(tabulate(
    df_iqr_scores[['service','TP','FP','FN','Precision','Recall','F1','FPR']],
    headers='keys', tablefmt='rounded_outline', showindex=False
))
avg_iqr = df_iqr_scores[['Precision','Recall','F1']].mean()
print(f'\nAverage  Precision={avg_iqr["Precision"]:.3f}  Recall={avg_iqr["Recall"]:.3f}  F1={avg_iqr["F1"]:.3f}')

In [ ]:
# ── Visualise Z-Score detection on memory leak anomaly ─────────────────────────
df_zv = results_zscore['order-service'].copy()
df_zv['hour'] = (df_zv['timestamp'] - SIM_START).dt.total_seconds() / 3600

# Tuesday window (hours 24–36): where the memory leak is
win = (df_zv['hour'] >= 24) & (df_zv['hour'] <= 36)
dz  = df_zv[win]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Phase 3 — Rolling Z-Score Detection: Memory Leak (order-service, Tuesday)',
             fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(dz['hour'], dz['mem_pct'], color='#9b59b6', linewidth=0.9, label='Memory %')
roll_mean_m = dz['mem_pct'].rolling(ZSCORE_WINDOW, min_periods=10).mean()
roll_std_m  = dz['mem_pct'].rolling(ZSCORE_WINDOW, min_periods=10).std()
ax.fill_between(dz['hour'],
                roll_mean_m - 3 * roll_std_m,
                roll_mean_m + 3 * roll_std_m,
                alpha=0.2, color='blue', label='±3σ band (rolling)')
ax.plot(dz['hour'], roll_mean_m, color='blue', linewidth=1, linestyle='--', label='Rolling mean')
# Mark detected anomalies
anom = dz[dz['zscore_pred'] == 1]
ax.scatter(anom['hour'], anom['mem_pct'], color='red', s=12, zorder=5, label='Z-Score anomaly detected')
ax.set_ylabel('Memory %')
ax.set_title('Memory metric — Z-Score band adapts to slow drift, catches the leak')
ax.legend(fontsize=8)

ax2 = axes[1]
ax2.plot(dz['hour'], dz['zscore_mem_pct'], color='#9b59b6', linewidth=0.9, label='Z-Score (memory)')
ax2.axhline(ZSCORE_THRESH, color='red', linestyle='--', linewidth=1.5, label=f'+{ZSCORE_THRESH}σ threshold')
ax2.axhline(-ZSCORE_THRESH, color='red', linestyle='--', linewidth=1.5, label=f'-{ZSCORE_THRESH}σ threshold')
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.fill_between(dz['hour'],
                 dz['is_anomaly'] * ZSCORE_THRESH * 2 - ZSCORE_THRESH,
                 -ZSCORE_THRESH, where=dz['is_anomaly'] == 1,
                 alpha=0.15, color='red', label='True anomaly window')
ax2.set_ylabel('Z-Score')
ax2.set_xlabel('Hour of week')
ax2.set_title('Z-Score signal — exceeds ±3σ when anomaly occurs')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/phase3_zscore.png', dpi=120, bbox_inches='tight')
plt.show()
print('💡 Z-Score with rolling window correctly identifies the gradual memory leak')
print('   that the static threshold MISSED in Phase 2.')

---
## Phase 4 — Isolation Forest: Multivariate Anomaly Detection 🌲

### 📖 Concept: Why Multivariate Detection Matters

Many anomalies are **invisible in any single metric** but clearly visible when multiple metrics are considered together:

```
  CPU = 72%       → Normal alone
  Latency = 480ms → Normal alone  
  Errors  = 4.5%  → Normal alone
  GC = 85ms       → Normal alone
  
  But ALL FOUR elevated SIMULTANEOUSLY → Anomaly!
```

**Isolation Forest** works by randomly partitioning the feature space. Points that are *easy to isolate* (require fewer partitions) are anomalies — they live in sparse regions of the feature space away from the normal cluster.

> Time complexity: O(n log n) · Works well with 5–50 features · No distance metric needed

In [ ]:
# ── Isolation Forest on all 8 metrics ─────────────────────────────────────────
IF_FEATURES = ['cpu_pct', 'mem_pct', 'rps', 'error_rate_pct',
               'p99_latency_ms', 'net_bytes_out_kb', 'gc_pause_ms', 'pod_restarts']
CONTAMINATION = 0.03   # expected fraction of anomalies (~3% of data)

results_iforest = {}
iforest_scores  = []

for svc in SVC_NAMES:
    df = dfs[svc].copy()

    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(df[IF_FEATURES].fillna(0))

    # Train Isolation Forest
    iforest = IsolationForest(
        n_estimators=200,
        contamination=CONTAMINATION,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )
    preds = iforest.fit_predict(X)   # -1 = anomaly, 1 = normal
    scores = iforest.score_samples(X)   # lower = more anomalous

    df['if_pred']  = (preds == -1).astype(int)
    df['if_score'] = scores
    results_iforest[svc] = df
    iforest_scores.append(detection_metrics(df['is_anomaly'], df['if_pred'], 'IsolationForest', svc))
    print(f'  {svc:25s}: {df["if_pred"].sum():>4} anomalies detected')

df_if_scores = pd.DataFrame(iforest_scores)
print()
print('Isolation Forest Detection Results:')
print(tabulate(
    df_if_scores[['service','TP','FP','FN','Precision','Recall','F1','FPR']],
    headers='keys', tablefmt='rounded_outline', showindex=False
))
avg_if = df_if_scores[['Precision','Recall','F1']].mean()
print(f'\nAverage  Precision={avg_if["Precision"]:.3f}  Recall={avg_if["Recall"]:.3f}  F1={avg_if["F1"]:.3f}')

In [ ]:
# ── Visualise: Anomaly Score Heatmap & Feature Importance ─────────────────────
svc_viz = 'order-service'
df_ifv  = results_iforest[svc_viz].copy()
df_ifv['hour'] = (df_ifv['timestamp'] - SIM_START).dt.total_seconds() / 3600

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f'Phase 4 — Isolation Forest: {svc_viz}',
             fontsize=12, fontweight='bold')

# Panel 1: Anomaly score over time
ax = axes[0]
ax.plot(df_ifv['hour'], df_ifv['if_score'],
        color='#2c3e50', linewidth=0.6, alpha=0.8, label='IF anomaly score')
# Shade anomaly windows
for (s, d, svc, _, _) in ANOMALY_EVENTS:
    if svc == svc_viz:
        ax.axvspan(s/60, (s+d)/60, alpha=0.2, color='red')
threshold_score = np.percentile(df_ifv['if_score'], CONTAMINATION * 100)
ax.axhline(threshold_score, color='red', linestyle='--',
           linewidth=1.5, label=f'Anomaly threshold (contamination={CONTAMINATION})')
ax.set_ylabel('IF Score (lower = more anomalous)')
ax.set_title('Isolation Forest Score — Drops During Injected Anomalies')
ax.legend(fontsize=8)
ax.set_xlim(0, SIM_DAYS * 24)

# Panel 2: Multi-metric heatmap (sampled hourly)
ax2 = axes[1]
hourly = df_ifv.groupby(df_ifv['hour'].astype(int))[IF_FEATURES[:6]].mean()
scaler_viz = MinMaxScaler()
hourly_scaled = pd.DataFrame(
    scaler_viz.fit_transform(hourly),
    columns=hourly.columns, index=hourly.index
)
im = ax2.imshow(hourly_scaled.T, aspect='auto', cmap='RdYlGn_r',
                extent=[0, SIM_DAYS*24, -0.5, len(IF_FEATURES[:6])-0.5],
                vmin=0, vmax=1)
ax2.set_yticks(range(len(IF_FEATURES[:6])))
ax2.set_yticklabels(IF_FEATURES[:6], fontsize=8)
ax2.set_title('Metric Heatmap — Red = high value, Green = low value (hourly aggregated)')
plt.colorbar(im, ax=ax2, shrink=0.8, label='Normalised value')

# Panel 3: Ground truth vs IF prediction
ax3 = axes[2]
ax3.fill_between(df_ifv['hour'], df_ifv['is_anomaly'],
                 alpha=0.6, color=COLORS['anomaly'], label='Ground truth')
ax3.fill_between(df_ifv['hour'], df_ifv['if_pred'] * 0.6,
                 alpha=0.5, color=COLORS['warning'], label='IF prediction')
ax3.set_ylabel('Anomaly')
ax3.set_xlabel('Hour of week')
ax3.set_title('Ground Truth vs Isolation Forest Predictions')
ax3.legend(fontsize=8)
day_ticks = list(range(0, (SIM_DAYS+1)*24, 24))
ax3.set_xticks(day_ticks)
ax3.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun',''])

plt.tight_layout()
plt.savefig('/tmp/phase4_iforest.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2D scatter: visualise multivariate separation ─────────────────────────────
# Project the 8 features onto the 2 most discriminating dimensions
from sklearn.decomposition import PCA

svc_sc = 'order-service'
df_sc  = results_iforest[svc_sc].copy()
X_sc   = StandardScaler().fit_transform(df_sc[IF_FEATURES].fillna(0))
pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Phase 4 — Isolation Forest: Feature Space ({svc_sc})',
             fontsize=12, fontweight='bold')

# Panel 1: Colour by ground truth
ax = axes[0]
normal_mask = df_sc['is_anomaly'] == 0
ax.scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1],
           c=COLORS['normal'], s=2, alpha=0.3, label='Normal')
ax.scatter(X_pca[~normal_mask, 0], X_pca[~normal_mask, 1],
           c=COLORS['anomaly'], s=8, alpha=0.8, label='True Anomaly')
ax.set_title('Ground Truth Labels in PCA Space')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(fontsize=9)

# Panel 2: Colour by IF prediction
ax2 = axes[1]
pred_normal = df_sc['if_pred'] == 0
ax2.scatter(X_pca[pred_normal, 0], X_pca[pred_normal, 1],
            c=COLORS['normal'], s=2, alpha=0.3, label='IF: Normal')
ax2.scatter(X_pca[~pred_normal, 0], X_pca[~pred_normal, 1],
            c=COLORS['anomaly'], s=8, alpha=0.8, label='IF: Anomaly')
ax2.set_title('Isolation Forest Predictions in PCA Space')
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/phase4_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print('💡 Anomalous points cluster in the periphery of the feature space.')
print('   Isolation Forest correctly separates them without needing labels.')

---
## Phase 5 — DBSCAN: Density-Based Anomaly Detection 🔵

### 📖 Concept: Anomalies as Low-Density Points

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) defines clusters as dense regions and marks sparse, isolated points as **noise** — and in anomaly detection, noise = anomaly.

- Points in **dense regions** = normal behaviour
- Points labelled **-1 (noise)** by DBSCAN = anomalies

DBSCAN is especially useful for detecting **novel anomaly patterns** that don't look like any known cluster, and for discovering **which anomalies cluster together** (same root cause?).

Key parameters: `eps` (neighbourhood radius), `min_samples` (density threshold)

In [ ]:
# ── DBSCAN anomaly detection ───────────────────────────────────────────────────
DBSCAN_EPS        = 0.55    # neighbourhood radius in normalised space
DBSCAN_MINSAMPLES = 8       # minimum points to form a cluster
DBSCAN_FEATURES   = ['cpu_pct', 'mem_pct', 'p99_latency_ms', 'error_rate_pct', 'gc_pause_ms']

results_dbscan = {}
dbscan_scores  = []

for svc in SVC_NAMES:
    df = dfs[svc].copy()

    # Subsample for speed (every 3rd point = 20min resolution for DBSCAN)
    sample_idx = df.index[::3]
    df_sample  = df.loc[sample_idx].copy()

    X = StandardScaler().fit_transform(df_sample[DBSCAN_FEATURES].fillna(0))
    db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINSAMPLES, n_jobs=-1).fit(X)

    df_sample = df_sample.copy()
    df_sample['db_cluster'] = db.labels_
    df_sample['db_pred']    = (db.labels_ == -1).astype(int)  # -1 = noise = anomaly

    # Propagate labels back to full resolution via forward-fill
    df['db_cluster'] = np.nan
    df['db_pred']    = 0
    df.loc[sample_idx, 'db_cluster'] = db.labels_.astype(float)
    df.loc[sample_idx, 'db_pred']    = df_sample['db_pred'].values
    df['db_pred'] = df['db_pred'].ffill().fillna(0).astype(int)

    results_dbscan[svc] = df
    dbscan_scores.append(detection_metrics(df['is_anomaly'], df['db_pred'], 'DBSCAN', svc))

    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise    = (db.labels_ == -1).sum()
    print(f'  {svc:25s}: {n_clusters} clusters, {n_noise} noise points (anomalies)')

df_db_scores = pd.DataFrame(dbscan_scores)
print()
print('DBSCAN Detection Results:')
print(tabulate(
    df_db_scores[['service','TP','FP','FN','Precision','Recall','F1']],
    headers='keys', tablefmt='rounded_outline', showindex=False
))
avg_db = df_db_scores[['Precision','Recall','F1']].mean()
print(f'\nAverage  Precision={avg_db["Precision"]:.3f}  Recall={avg_db["Recall"]:.3f}  F1={avg_db["F1"]:.3f}')

In [ ]:
# ── DBSCAN cluster visualisation ───────────────────────────────────────────────
svc_db = 'inventory-service'
df_db  = dfs[svc_db].copy()
sample_idx_db = df_db.index[::3]
df_sample_db  = df_db.loc[sample_idx_db].copy()
X_db = StandardScaler().fit_transform(df_sample_db[DBSCAN_FEATURES].fillna(0))
db2  = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINSAMPLES).fit(X_db)
df_sample_db = df_sample_db.copy()
df_sample_db['db_label'] = db2.labels_

# PCA for 2D projection
pca_db   = PCA(n_components=2, random_state=42)
X_pca_db = pca_db.fit_transform(X_db)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Phase 5 — DBSCAN Cluster Map ({svc_db})',
             fontsize=12, fontweight='bold')

# Panel 1: DBSCAN clusters
ax = axes[0]
unique_labels = sorted(set(db2.labels_))
cluster_cmap  = plt.cm.tab20
for label in unique_labels:
    mask = db2.labels_ == label
    if label == -1:
        ax.scatter(X_pca_db[mask, 0], X_pca_db[mask, 1],
                   c='red', s=20, zorder=5, marker='x', label='Noise (Anomaly)')
    else:
        ax.scatter(X_pca_db[mask, 0], X_pca_db[mask, 1],
                   c=[cluster_cmap(label / max(1, max(unique_labels)))],
                   s=4, alpha=0.4, label=f'Cluster {label}')
ax.set_title('DBSCAN: Noise Points = Anomalies')
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')
ax.legend(fontsize=7, markerscale=2, loc='upper right')

# Panel 2: anomaly score over time
ax2 = axes[1]
h_db = (df_sample_db['timestamp'] - SIM_START).dt.total_seconds() / 3600
normal_db = df_sample_db['db_label'] >= 0
ax2.scatter(h_db[normal_db],  np.zeros(normal_db.sum()),
            c=COLORS['normal'], s=3, alpha=0.3, label='Normal cluster')
ax2.scatter(h_db[~normal_db], np.ones((~normal_db).sum()),
            c=COLORS['anomaly'], s=15, alpha=0.9, label='Noise = Anomaly')
for (s, d, svc, _, _) in ANOMALY_EVENTS:
    if svc == svc_db:
        ax2.axvspan(s/60, (s+d)/60, alpha=0.15, color='red')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Normal', 'Anomaly'])
ax2.set_xlabel('Hour of week')
ax2.set_title('DBSCAN Anomaly Timeline — Red bands = true anomaly windows')
ax2.legend(fontsize=8)
ax2.set_xlim(0, SIM_DAYS * 24)
ax2.set_xticks(range(0, (SIM_DAYS+1)*24, 24))
ax2.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun',''])

plt.tight_layout()
plt.savefig('/tmp/phase5_dbscan.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Phase 6 — LSTM Autoencoder: Deep Learning for Temporal Anomalies 🧠

### 📖 Concept: Reconstruction Error as Anomaly Score

An **autoencoder** learns to compress normal data into a latent representation and reconstruct it. When an anomaly occurs, the reconstruction fails — the **reconstruction error** spikes.

**LSTM (Long Short-Term Memory)** layers capture **temporal dependencies** — patterns that span multiple timesteps:

```
  Input (window of T timesteps)
       │
  Encoder: LSTM → LSTM → Dense (latent)
       │
  Decoder: Dense → LSTM → LSTM
       │
  Output (reconstructed window)
       │
  Reconstruction Error = MAE(input, output)
  If error > threshold → ANOMALY
```

This catches **gradual drifts** and **sequential pattern violations** invisible to point-in-time methods.

> ⏱️ Training takes ~2–3 minutes. This is intentional — it demonstrates real ML workflow.

In [ ]:
# ── LSTM Autoencoder data preparation ─────────────────────────────────────────
LSTM_SVC       = 'order-service'   # focus on the service with most anomaly types
LSTM_FEATURES  = ['cpu_pct', 'mem_pct', 'p99_latency_ms', 'error_rate_pct', 'gc_pause_ms']
LSTM_WINDOW    = 30     # look-back window: 30 minutes
LSTM_STEP      = 1      # step size between windows

df_lstm = dfs[LSTM_SVC].copy()

# Normalise features to [0, 1]
scaler_lstm = MinMaxScaler()
data_scaled = scaler_lstm.fit_transform(df_lstm[LSTM_FEATURES].fillna(0))

# Split: train on first 5 days (Mon–Fri), test on full week
train_size = 5 * 24 * 60    # 7200 minutes
train_data = data_scaled[:train_size]
test_data  = data_scaled

def create_sequences(data, window, step=1):
    """Create sliding window sequences for LSTM input."""
    sequences = []
    for i in range(0, len(data) - window, step):
        sequences.append(data[i:i+window])
    return np.array(sequences)

X_train = create_sequences(train_data, LSTM_WINDOW, LSTM_STEP)
X_test  = create_sequences(test_data,  LSTM_WINDOW, LSTM_STEP)

print(f'Training sequences : {X_train.shape}  ({X_train.shape[0]:,} windows × {LSTM_WINDOW}min × {len(LSTM_FEATURES)} features)')
print(f'Test sequences     : {X_test.shape}')
print(f'Train on: days 1–5 (normal data only to teach the model what normal looks like)')
print(f'Test on : full 7 days (including anomalous periods)')

In [ ]:
# ── Build the LSTM Autoencoder ────────────────────────────────────────────────

n_features  = len(LSTM_FEATURES)
seq_len     = LSTM_WINDOW
LATENT_DIM  = 16

# ── Encoder ───────────────────────────────────────────────────────────────────
inputs  = keras.Input(shape=(seq_len, n_features))
x       = layers.LSTM(64, return_sequences=True, name='enc_lstm1')(inputs)
x       = layers.Dropout(0.1)(x)
x       = layers.LSTM(32, return_sequences=False, name='enc_lstm2')(x)
latent  = layers.Dense(LATENT_DIM, activation='relu', name='latent')(x)

# ── Decoder ───────────────────────────────────────────────────────────────────
x       = layers.Dense(32, activation='relu')(latent)
x       = layers.RepeatVector(seq_len)(x)
x       = layers.LSTM(32, return_sequences=True, name='dec_lstm1')(x)
x       = layers.Dropout(0.1)(x)
x       = layers.LSTM(64, return_sequences=True, name='dec_lstm2')(x)
outputs = layers.TimeDistributed(layers.Dense(n_features, activation='sigmoid'), name='output')(x)

autoencoder = keras.Model(inputs, outputs, name='LSTM_Autoencoder')
autoencoder.compile(optimizer='adam', loss='mae')

autoencoder.summary()
print(f'\nTotal parameters: {autoencoder.count_params():,}')

In [ ]:
# ── Train the autoencoder ─────────────────────────────────────────────────────
# Train ONLY on normal data (first 5 days).
# The model learns to reconstruct normal patterns.
# It will fail on anomalous patterns → high reconstruction error.

print('Training LSTM Autoencoder on 5 days of normal data...')
print('(This teaches the model what normal operational patterns look like)\n')

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
)

history = autoencoder.fit(
    X_train, X_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f'\nTraining complete. Best val_loss: {min(history.history["val_loss"]):.5f}')

In [ ]:
# ── Compute reconstruction error & detect anomalies ───────────────────────────
print('Computing reconstruction errors on full test set...')

X_pred     = autoencoder.predict(X_test, batch_size=128, verbose=0)
recon_err  = np.mean(np.abs(X_test - X_pred), axis=(1, 2))  # MAE per window

# Set threshold: mean + 3 std of reconstruction error on TRAINING data
X_train_pred  = autoencoder.predict(X_train, batch_size=128, verbose=0)
train_err     = np.mean(np.abs(X_train - X_train_pred), axis=(1, 2))
lstm_threshold = train_err.mean() + 3 * train_err.std()

print(f'Reconstruction error threshold: {lstm_threshold:.5f}')
print(f'  Train error: mean={train_err.mean():.5f}  std={train_err.std():.5f}')
print(f'  Test  error: mean={recon_err.mean():.5f}  std={recon_err.std():.5f}')

# Map window predictions back to original time indices
lstm_pred_full = np.zeros(N_POINTS, dtype=int)
lstm_err_full  = np.full(N_POINTS, np.nan)
for i, err in enumerate(recon_err):
    start_idx = i * LSTM_STEP
    end_idx   = start_idx + LSTM_WINDOW
    if err > lstm_threshold:
        lstm_pred_full[start_idx:end_idx] = 1
    lstm_err_full[start_idx] = err

df_lstm['lstm_pred']  = lstm_pred_full
df_lstm['lstm_err']   = lstm_err_full

lstm_score = detection_metrics(df_lstm['is_anomaly'], df_lstm['lstm_pred'], 'LSTM-AE', LSTM_SVC)
print(f'\nLSTM Autoencoder Results ({LSTM_SVC}):')
print(f'  Precision={lstm_score["Precision"]:.3f}  Recall={lstm_score["Recall"]:.3f}  '
      f'F1={lstm_score["F1"]:.3f}  FPR={lstm_score["FPR"]:.4f}')

In [ ]:
# ── LSTM detection visualisation ──────────────────────────────────────────────
df_lv = df_lstm.copy()
df_lv['hour'] = (df_lv['timestamp'] - SIM_START).dt.total_seconds() / 3600

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f'Phase 6 — LSTM Autoencoder: {LSTM_SVC} — Reconstruction Error as Anomaly Score',
             fontsize=12, fontweight='bold')

# Panel 1: Training loss curve
ax = axes[0]
ax.plot(history.history['loss'],     color='#3498db', label='Train loss', linewidth=1.5)
ax.plot(history.history['val_loss'], color='#e74c3c', label='Val loss',   linewidth=1.5)
ax.set_ylabel('MAE Loss')
ax.set_xlabel('Epoch')
ax.set_title('Training Convergence — Model Learned Normal Behaviour')
ax.legend(fontsize=9)

# Panel 2: Reconstruction error over time
ax2 = axes[1]
valid_err = ~np.isnan(df_lv['lstm_err'])
ax2.plot(df_lv[valid_err]['hour'], df_lv[valid_err]['lstm_err'],
         color='#2c3e50', linewidth=0.7, alpha=0.8, label='Reconstruction error')
ax2.axhline(lstm_threshold, color='red', linestyle='--', linewidth=2,
            label=f'Anomaly threshold ({lstm_threshold:.4f})')
ax2.fill_between(df_lv[valid_err]['hour'],
                 df_lv[valid_err]['lstm_err'], lstm_threshold,
                 where=df_lv[valid_err]['lstm_err'] > lstm_threshold,
                 alpha=0.4, color='red', label='Above threshold')
for (s, d, svc, _, _) in ANOMALY_EVENTS:
    if svc == LSTM_SVC:
        ax2.axvspan(s/60, (s+d)/60, alpha=0.12, color='purple')
ax2.set_ylabel('Reconstruction MAE')
ax2.set_title('Reconstruction Error — Spikes During Anomalous Periods (purple = true anomalies)')
ax2.legend(fontsize=8)
ax2.set_xlim(0, SIM_DAYS * 24)

# Panel 3: Prediction vs ground truth
ax3 = axes[2]
ax3.fill_between(df_lv['hour'], df_lv['is_anomaly'],
                 alpha=0.6, color=COLORS['anomaly'], label='Ground truth')
ax3.fill_between(df_lv['hour'], df_lv['lstm_pred'] * 0.6,
                 alpha=0.5, color='#9b59b6', label='LSTM-AE prediction')
ax3.set_ylabel('Anomaly')
ax3.set_xlabel('Hour of week')
ax3.set_title('LSTM Autoencoder vs Ground Truth')
ax3.legend(fontsize=8)
ax3.set_xticks(range(0, (SIM_DAYS+1)*24, 24))
ax3.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun',''])

plt.tight_layout()
plt.savefig('/tmp/phase6_lstm.png', dpi=120, bbox_inches='tight')
plt.show()
print('💡 LSTM Autoencoder catches temporal pattern violations — gradual drifts')
print('   and sequential anomalies that point-in-time methods miss entirely.')

---
## Phase 7 — Kubernetes-Specific Anomaly Detection 🐳

### 📖 Concept: Infrastructure Anomalies in Kubernetes

Beyond application metrics, Kubernetes exposes infrastructure signals that are **unique to container orchestration**:

| Signal | What it indicates | Anomaly when... |
|--------|------------------|-----------------|
| `pod_restarts` | Pod crashed and was restarted | > 2 restarts in 5 minutes |
| `cpu_throttle_pct` | % of CPU quota time throttled | > 25% (pod needs more CPU limit) |
| `oom_kill_count` | Out-of-memory kills by kubelet | Any OOM kill is anomalous |
| `pending_pods` | Pods stuck in Pending state | > 0 for more than 2 minutes |
| `node_pressure` | Node memory/disk pressure events | Any pressure event |
| `container_start_time` | New container started | Churn rate > baseline |

We will:
1. Simulate K8s infrastructure metrics
2. Query live `kubectl` data from our lab cluster
3. Apply detection rules specific to Kubernetes semantics

In [ ]:
# ── Query live Kubernetes metrics ──────────────────────────────────────────────
# Pull real data from the cluster to enrich our analysis.

print('=== Live Kubernetes Cluster State ===')
print()

print('── Pod Status (aiops-lab9 namespace) ──')
run(f'kubectl get pods -n {NAMESPACE} -o wide')

print()
print('── Resource Usage (kubectl top) ──')
out_top, rc_top = run(f'kubectl top pods -n {NAMESPACE} 2>/dev/null || echo "metrics-server not available — simulating"')

print()
print('── Pod Describe: Check for Restart/OOM events on inventory-service ──')
run(f'kubectl describe pods -n {NAMESPACE} -l app=inventory-service | grep -A5 "Conditions\|Restart\|OOM\|Events" | head -40')

print()
print('── Node conditions ──')
run('kubectl get nodes -o custom-columns=NAME:.metadata.name,STATUS:.status.conditions[-1].type,REASON:.status.conditions[-1].reason')

In [ ]:
# ── Simulate 24 hours of Kubernetes infrastructure metrics ────────────────────
# (In production, these come from kube-state-metrics + cAdvisor + node-exporter)

K8S_MINUTES    = 24 * 60    # 1 day at 1-min resolution
k8s_timestamps = [datetime(2025, 6, 12, 0, 0) + timedelta(minutes=i) for i in range(K8S_MINUTES)]
k8s_t          = np.arange(K8S_MINUTES)

np.random.seed(99)

# Normal baseline
k8s_cpu_throttle  = np.clip(np.random.normal(5,  3,  K8S_MINUTES), 0, 100)  # % throttle
k8s_mem_pressure  = np.zeros(K8S_MINUTES, dtype=int)                          # node memory pressure events
k8s_oom_kills     = np.zeros(K8S_MINUTES, dtype=int)                          # OOM kills
k8s_pod_restarts  = np.zeros(K8S_MINUTES, dtype=int)                          # pod restarts
k8s_pending_pods  = np.zeros(K8S_MINUTES, dtype=int)                          # pending pod count
k8s_net_errors    = np.clip(np.random.poisson(0.5, K8S_MINUTES), 0, 5)        # network errors/min
k8s_is_anomaly    = np.zeros(K8S_MINUTES, dtype=int)

# ── Inject K8s-specific anomaly scenarios ─────────────────────────────────────
K8S_ANOMALIES = [
    # (start_min, dur, type,               description)
    (120,  30,  'cpu_throttle',      'order-service hitting CPU limit — needs right-sizing'),
    (300,  10,  'oom_kill',          'inventory-service OOMKilled — memory limit too low'),
    (480,  45,  'pod_churn',         'payment-service CrashLoopBackoff — 8 restarts'),
    (600,  20,  'node_pressure',     'Worker node memory pressure — kubelet evicting pods'),
    (720,  15,  'pending_pods',      'Scheduler can not place pods — resource exhaustion'),
    (900,  60,  'network_errors',    'CNI plugin errors — inter-pod communication failures'),
    (1200, 25,  'cpu_throttle',      'api-gateway CPU throttle during traffic spike'),
]

for (start, dur, atype, _) in K8S_ANOMALIES:
    end = min(start + dur, K8S_MINUTES)
    k8s_is_anomaly[start:end] = 1

    if atype == 'cpu_throttle':
        k8s_cpu_throttle[start:end] += np.random.uniform(40, 70, end-start)

    elif atype == 'oom_kill':
        k8s_oom_kills[start:start+3] = [1, 2, 1]
        k8s_pod_restarts[start:start+5] = [0, 1, 0, 1, 0]

    elif atype == 'pod_churn':
        for i in range(8):
            if start + i*5 < K8S_MINUTES:
                k8s_pod_restarts[start + i*5] = 1

    elif atype == 'node_pressure':
        k8s_mem_pressure[start:end] = 1
        k8s_pending_pods[start+5:end] = np.random.randint(1, 4, end-start-5)

    elif atype == 'pending_pods':
        k8s_pending_pods[start:end] = np.random.randint(2, 6, end-start)

    elif atype == 'network_errors':
        k8s_net_errors[start:end] += np.random.poisson(15, end-start)

k8s_cpu_throttle = np.clip(k8s_cpu_throttle, 0, 100)

df_k8s = pd.DataFrame({
    'timestamp':        k8s_timestamps,
    'cpu_throttle_pct': np.round(k8s_cpu_throttle, 2),
    'mem_pressure':     k8s_mem_pressure,
    'oom_kills':        k8s_oom_kills,
    'pod_restarts':     k8s_pod_restarts,
    'pending_pods':     k8s_pending_pods,
    'net_errors':       k8s_net_errors,
    'is_anomaly':       k8s_is_anomaly,
})

print(f'K8s metric dataset: {len(df_k8s):,} rows')
print(f'Anomalous minutes : {df_k8s["is_anomaly"].sum()} ({df_k8s["is_anomaly"].mean()*100:.1f}%)')

In [ ]:
# ── Kubernetes-specific detection rules ───────────────────────────────────────
# K8s anomaly detection uses SEMANTIC RULES specific to container behaviour.
# These are not statistical thresholds — they encode Kubernetes domain knowledge.

def k8s_detect(df):
    df = df.copy()
    flags = pd.DataFrame(index=df.index)

    # Rule 1: CPU throttle > 25% is always a concern (pod needs bigger CPU limit)
    flags['cpu_throttle'] = (df['cpu_throttle_pct'] > 25).astype(int)

    # Rule 2: ANY OOM kill = immediate anomaly (memory limit needs to be raised)
    flags['oom_kill'] = (df['oom_kills'] > 0).astype(int)

    # Rule 3: Pod restart rate — rolling 5-min window: more than 2 restarts
    restart_roll = df['pod_restarts'].rolling(5, min_periods=1).sum()
    flags['pod_churn'] = (restart_roll > 2).astype(int)

    # Rule 4: Any memory pressure event from kubelet
    flags['node_pressure'] = (df['mem_pressure'] > 0).astype(int)

    # Rule 5: Pending pods for > 2 consecutive minutes = scheduling failure
    pending_duration = (
        df['pending_pods'].rolling(2, min_periods=1).min() > 0
    ).astype(int)
    flags['pending_pods'] = pending_duration

    # Rule 6: Network errors spike — rolling Z-score > 4
    roll_mean = df['net_errors'].rolling(30, min_periods=5).mean()
    roll_std  = df['net_errors'].rolling(30, min_periods=5).std()
    z_net     = (df['net_errors'] - roll_mean) / (roll_std + 1e-9)
    flags['network_anomaly'] = (z_net > 4).astype(int)

    # Combined: anomaly if ANY rule fires
    df['k8s_pred']     = flags.any(axis=1).astype(int)
    df['k8s_reason']   = flags.idxmax(axis=1).where(flags.any(axis=1), None)
    for col in flags.columns:
        df[f'flag_{col}'] = flags[col]
    return df

df_k8s = k8s_detect(df_k8s)

k8s_score = detection_metrics(df_k8s['is_anomaly'], df_k8s['k8s_pred'], 'K8s-Rules', 'cluster')

print('Kubernetes-Specific Detection Results:')
print(tabulate([[k, v] for k, v in k8s_score.items()],
               headers=['Metric', 'Value'], tablefmt='rounded_outline'))

print()
print('Detected K8s anomaly types breakdown:')
for col in [c for c in df_k8s.columns if c.startswith('flag_')]:
    count = df_k8s[col].sum()
    print(f'  {col.replace("flag_",""):25s}: {count:>4} minutes flagged')

In [ ]:
# ── Kubernetes anomaly dashboard ───────────────────────────────────────────────
df_k8s['hour'] = (df_k8s['timestamp'] - df_k8s['timestamp'].iloc[0]).dt.total_seconds() / 3600

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle('Phase 7 — Kubernetes Infrastructure Anomaly Dashboard\n(24-hour window · Red bands = anomaly periods)',
             fontsize=12, fontweight='bold')

K8S_METRICS_PLOT = [
    ('cpu_throttle_pct',  'CPU Throttle %',        '#e74c3c',  25),
    ('pod_restarts',      'Pod Restarts',           '#c0392b',  1),
    ('oom_kills',         'OOM Kill Count',         '#8e44ad',  0.5),
    ('mem_pressure',      'Node Memory Pressure',   '#e67e22',  0.5),
    ('pending_pods',      'Pending Pods',           '#d35400',  0.5),
    ('net_errors',        'Network Errors/min',     '#2980b9',  5),
    ('k8s_pred',          'K8s Anomaly Prediction', '#27ae60',  0.5),
    ('is_anomaly',        'Ground Truth',           '#7f8c8d',  0.5),
]

for ax, (metric, label, color, thresh) in zip(axes.flatten(), K8S_METRICS_PLOT):
    ax.plot(df_k8s['hour'], df_k8s[metric], color=color, linewidth=0.8, alpha=0.9)
    if thresh:
        ax.axhline(thresh, color='red', linestyle=':', linewidth=1, alpha=0.6)
    # Shade true anomaly windows
    ax.fill_between(df_k8s['hour'], 0,
                    df_k8s['is_anomaly'] * df_k8s[metric].max() * 1.1,
                    where=df_k8s['is_anomaly'] == 1,
                    alpha=0.1, color='red')
    ax.set_ylabel(label, fontsize=8)
    ax.set_xlabel('Hour', fontsize=8)
    # Annotate anomaly types
    for (start, dur, atype, desc) in K8S_ANOMALIES:
        mid = (start + dur/2) / 60
        if mid < 24:
            ax.axvspan(start/60, (start+dur)/60, alpha=0.12, color='orange')

plt.tight_layout()
plt.savefig('/tmp/phase7_k8s.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print('K8s Anomaly Events Detected:')
event_rows = []
for (start, dur, atype, desc) in K8S_ANOMALIES:
    detected = df_k8s['k8s_pred'][start:start+dur].sum()
    event_rows.append([f'{start//60:02d}:{start%60:02d}', f'{dur}min', atype, desc, f'{detected}/{dur}'])
print(tabulate(event_rows, headers=['Time','Duration','Type','Description','Detected/Total'],
               tablefmt='rounded_outline'))

---
## Phase 8 — Method Comparison: Which Detector Wins? 🏆

### 📖 Concept: Choosing the Right Anomaly Detector

There is no universally best anomaly detector. The right choice depends on:

| Factor | Relevant Detector |
|--------|-------------------|
| Need to explain the decision | Threshold Rules, Z-Score |
| Gradual drifts / trends | Rolling Z-Score, LSTM |
| Multi-metric correlation | Isolation Forest, DBSCAN |
| Novel/unknown anomaly patterns | DBSCAN, LSTM Autoencoder |
| Real-time streaming (low latency) | Threshold, Z-Score |
| K8s container semantics | Rule-based K8s detector |
| High data volume, low label cost | Isolation Forest |

We compare all methods head-to-head on the **order-service** dataset.

In [ ]:
# ── Aggregate all detection results for order-service ────────────────────────
svc_cmp = 'order-service'

comparison_data = [
    ('Static Threshold',    results_thresh[svc_cmp]['thresh_pred']),
    ('Rolling Z-Score',     results_zscore[svc_cmp]['zscore_pred']),
    ('IQR',                 results_iqr[svc_cmp]['iqr_pred']),
    ('Isolation Forest',    results_iforest[svc_cmp]['if_pred']),
    ('DBSCAN',              results_dbscan[svc_cmp]['db_pred']),
    ('LSTM Autoencoder',    df_lstm['lstm_pred']),
]

y_true = dfs[svc_cmp]['is_anomaly']

comparison_rows = []
for name, preds in comparison_data:
    s = detection_metrics(y_true, preds, name, svc_cmp)
    comparison_rows.append(s)

df_compare = pd.DataFrame(comparison_rows)

print(f'METHOD COMPARISON — {svc_cmp}')
print(tabulate(
    df_compare[['method','TP','FP','FN','Precision','Recall','F1','FPR']].sort_values('F1', ascending=False),
    headers=['Method','TP','FP','FN','Precision','Recall','F1','FPR'],
    tablefmt='rounded_outline', showindex=False
))

best = df_compare.sort_values('F1', ascending=False).iloc[0]
print(f'\n🏆 Best overall F1 score: {best["method"]} (F1={best["F1"]:.3f})')

In [ ]:
# ── Comparison visualisation ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Phase 8 — Anomaly Detection Methods: Head-to-Head Comparison',
             fontsize=13, fontweight='bold')

methods = df_compare['method'].tolist()
bar_colors = ['#e74c3c','#3498db','#2ecc71','#9b59b6','#e67e22','#1abc9c']

# Panel 1: F1 Score
ax = axes[0, 0]
bars = ax.barh(df_compare.sort_values('F1')['method'],
               df_compare.sort_values('F1')['F1'],
               color=bar_colors[:len(methods)], edgecolor='white', height=0.6)
ax.set_xlabel('F1 Score')
ax.set_title('F1 Score (higher = better)', fontweight='bold')
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, df_compare.sort_values('F1')['F1']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

# Panel 2: Precision vs Recall scatter
ax2 = axes[0, 1]
for i, row in df_compare.iterrows():
    ax2.scatter(row['Recall'], row['Precision'],
                color=bar_colors[i % len(bar_colors)], s=120, zorder=5)
    ax2.annotate(row['method'],
                 (row['Recall'], row['Precision']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
ax2.set_xlabel('Recall (sensitivity — fewer misses)')
ax2.set_ylabel('Precision (fewer false alarms)')
ax2.set_title('Precision vs Recall Trade-off', fontweight='bold')
ax2.set_xlim(-0.05, 1.05)
ax2.set_ylim(-0.05, 1.05)
ax2.axline((0, 0), (1, 1), linestyle='--', color='gray', alpha=0.4)

# Panel 3: False Positive Rate comparison
ax3 = axes[1, 0]
fpr_sorted = df_compare.sort_values('FPR', ascending=True)
bars3 = ax3.barh(fpr_sorted['method'], fpr_sorted['FPR'],
                 color=bar_colors[:len(methods)], edgecolor='white', height=0.6)
ax3.set_xlabel('False Positive Rate (lower = fewer false alarms)')
ax3.set_title('False Positive Rate (lower = better)', fontweight='bold')
for bar, val in zip(bars3, fpr_sorted['FPR']):
    ax3.text(val + 0.0002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=8)

# Panel 4: TP/FP/FN grouped bar
ax4 = axes[1, 1]
x4 = np.arange(len(methods))
w  = 0.25
ax4.bar(x4 - w, df_compare['TP'], w, label='TP (correct detections)', color='#27ae60', alpha=0.85)
ax4.bar(x4,     df_compare['FP'], w, label='FP (false alarms)',        color='#e67e22', alpha=0.85)
ax4.bar(x4 + w, df_compare['FN'], w, label='FN (missed anomalies)',    color='#e74c3c', alpha=0.85)
ax4.set_xticks(x4)
ax4.set_xticklabels([m.replace(' ', '\n') for m in methods], fontsize=7)
ax4.set_title('TP / FP / FN Breakdown per Method', fontweight='bold')
ax4.set_ylabel('Minute count')
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/phase8_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print('📊 INTERPRETATION GUIDE:')
print('  • Static Threshold : Fast, explainable, but many FPs + missed slow drifts')
print('  • Z-Score          : Adapts to baseline, catches gradual changes')
print('  • IQR              : Robust to outliers, good for skewed distributions')
print('  • Isolation Forest : Best multivariate detection, no labels needed')
print('  • DBSCAN           : Finds novel cluster patterns, discovers anomaly types')
print('  • LSTM Autoencoder : Catches temporal pattern violations, best for slow drifts')

In [ ]:
# ── Bonus: Ensemble detector — majority vote ──────────────────────────────────
# Combine all detectors: flag as anomaly if MAJORITY agree.
# This dramatically reduces false positives while preserving recall.

votes = pd.DataFrame({
    'threshold':  results_thresh[svc_cmp]['thresh_pred'],
    'zscore':     results_zscore[svc_cmp]['zscore_pred'],
    'iqr':        results_iqr[svc_cmp]['iqr_pred'],
    'iforest':    results_iforest[svc_cmp]['if_pred'],
    'dbscan':     results_dbscan[svc_cmp]['db_pred'],
    'lstm':       df_lstm['lstm_pred'],
})

votes.index = dfs[svc_cmp].index
vote_sum     = votes.sum(axis=1)

# Different voting thresholds
for min_votes in [1, 2, 3, 4]:
    ensemble_pred  = (vote_sum >= min_votes).astype(int)
    ensemble_score = detection_metrics(y_true, ensemble_pred, f'Ensemble(≥{min_votes}/6)', svc_cmp)
    print(f'  Ensemble ≥{min_votes}/6 votes: '
          f'F1={ensemble_score["F1"]:.3f}  '
          f'Precision={ensemble_score["Precision"]:.3f}  '
          f'Recall={ensemble_score["Recall"]:.3f}  '
          f'FPR={ensemble_score["FPR"]:.4f}')

print()
print('💡 Ensemble ≥2/6 typically gives the best F1 by combining strengths of all detectors.')
print('   Ensemble ≥3/6 trades some recall for much lower false positive rate.')

---
## Phase 9 — Cleanup 🧹

Remove all Kubernetes resources created in this lab.

In [ ]:
# ── Delete the lab namespace ───────────────────────────────────────────────────
print('Deleting namespace aiops-lab9...')
out, rc = run('kubectl delete namespace aiops-lab9 --ignore-not-found=true')

if rc == 0:
    print('\n✅ Namespace aiops-lab9 deleted. All resources removed.')
else:
    print('\n⚠️  Verify manually: kubectl get all -n aiops-lab9')

---
## 📚 Lab Recap & Key Takeaways

### What You Built — End-to-End

```
  7-day metric dataset (50,400 observations, 10 injected anomalies)
         │
         ├──▶ Phase 2: Static Threshold    → Baseline, simple, many FPs
         ├──▶ Phase 3: Z-Score + IQR       → Dynamic baseline, catches drifts
         ├──▶ Phase 4: Isolation Forest    → Multivariate, no labels needed
         ├──▶ Phase 5: DBSCAN             → Density-based, finds novel patterns
         ├──▶ Phase 6: LSTM Autoencoder   → Temporal sequences, slow anomalies
         ├──▶ Phase 7: K8s Rules          → Container-semantic detection
         └──▶ Phase 8: Ensemble (majority vote) → Best of all worlds
```

### The 5 Core Principles of AIOps Anomaly Detection

| # | Principle | What you saw in the lab |
|---|-----------|-------------------------|
| 1 | **Static thresholds are a starting point, not a solution** | Phase 2 showed FPs on normal peaks and FNs on memory leaks |
| 2 | **Context changes what is normal** | Rolling window Z-score adapted to diurnal patterns |
| 3 | **Anomalies are multivariate** | Isolation Forest found anomalies invisible in any single metric |
| 4 | **Temporal patterns carry information** | LSTM caught gradual drifts that point-in-time methods missed |
| 5 | **Ensembles beat any single method** | Majority voting reduced FPR while preserving recall |

### Discussion Questions

1. **Seasonality**: How would you modify the rolling Z-score to handle **weekly** seasonality (same hour, different day of week)?
2. **Concept drift**: The LSTM was trained on one week of data. What happens if traffic patterns change permanently (e.g., new product launch doubles baseline load)? How do you re-train?
3. **Label scarcity**: Isolation Forest and DBSCAN are unsupervised. How would you incorporate the occasional *labelled* anomaly from a past incident to improve supervised detection?
4. **Streaming**: These methods were applied to historical data. How would you deploy the Isolation Forest for **real-time** detection on a Kafka stream?
5. **K8s at scale**: With 500 microservices, how do you prioritise which anomalies to surface first?

### Tools Used in Production

| What we built | Production equivalent |
|---------------|-----------------------|
| Rolling Z-Score | Prometheus `deriv()` + Alertmanager |
| Isolation Forest | Elasticsearch ML / Datadog Watchdog |
| LSTM Autoencoder | AWS Lookout for Metrics / Azure Metrics Advisor |
| K8s rules | Prometheus Operator + Kube-state-metrics |
| Ensemble | IBM Watson AIOps / Moogsoft / BigPanda |

### Further Reading

- [Google SRE — Practical Alerting](https://sre.google/sre-book/practical-alerting/)
- [Prometheus Alerting Best Practices](https://prometheus.io/docs/practices/alerting/)
- [Isolation Forest Paper (Liu et al. 2008)](https://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/icdm08b.pdf)
- [LSTMAE for Anomaly Detection (Malhotra et al.)](https://arxiv.org/abs/1607.00148)
- [Kube-state-metrics docs](https://github.com/kubernetes/kube-state-metrics)

---

```
✅  Lab 9 Complete — AIOps Session 9: Anomaly Detection in IT Operations
```